# 6.1 字元切詞有什麼成本？


「小小貓」和「cat」各三個文字元素，存成UTF-8卻各要9與3byte；🙂是一個圖案，需4byte。原文佔幾個模型位置，要先知道拆分單位。

token是模型一次處理的片段，tokenizer將原文拆成片段並轉ID；decoder把ID轉回文字，能否完整還原原文取決於字表與前處理規則。「切詞」不保證切出人類的詞，可以是一字、多字或字的一部分bytes；下面先分碼點與byte。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
texts = ["小小貓", "cat", "🙂"]
for text in texts:
    raw = text.encode("utf-8")
    print(text, "Unicode碼點", len(text), "UTF-8 bytes", len(raw))

輸入三段文字，`.encode("utf-8")` 把字串轉為這種儲存編碼的byte序列。三行應分別看到3與9、3與3、1與4。兩種三字文本，實際存儲bytes不同；🙂顯示一個圖案卻需四byte。一字元一token時前兩段都是三格，一byte一token時則九格與三格，這些單位不能互換。

為何模型成本受影響？[3.6](https://birdhackor.github.io/tiny-perceptron-vlm/3.6.html) 的注意力權重有T×T格，T是token序列長度；從三格變九格，這張表便從9格變81格。字表則影響 [2.1](https://birdhackor.github.io/tiny-perceptron-vlm/2.1.html) 的V×D參數，V是可用token種類數。較細拆法常讓V較小、T較長，較粗拆法相反，不能只看字表小就說全模型省。

Unicode碼點是文字標準中的編號，組合重音或emoji可能用多碼點。三格變九格，注意力表從9格變81格；較細單位又常需要较少種類V。長度與字表要一起看。完整256種byte可表示未見字，但中文序列较長，下一節合常見片段。

練習只加入 `"cat🙂"`，先預測Unicode碼點4、UTF-8 bytes7，再執行核對。再把英文cat換成貓，應是2與7：碼點減少，byte數卻一樣。你可以看到「幾字」不能直接當成「模型處理幾token」，還得說明採用哪種拆法。

<details>
<summary>補充：實作約定與原始紀錄</summary>

「能表示原文」和「模型會生成合法文字」也要分開。[T.4的中文詩實跑](https://birdhackor.github.io/tiny-perceptron-vlm/training.html#T.4)給一筆驗證詩的開頭`渡漢江\n李頻\n嶺外`，實際新生成部分是`孟沙，天夜天天天天春�`；這裡的`\n`表示原文中的換行。提示取至多24個byte，且只在完整Unicode碼點後結束，沒有把半個中文字交給模型。生成卻是逐byte選候選，最多寫32個新byte：這筆最後只寫到下一字的前兩個byte就達到上限，解碼便顯示替代符號`�`。中間的重複字另是一個品質問題，不能靠補齊末字解決。相同的32格讓英文常能寫32個字母或標點，中文卻通常只夠約十個字；看生成長短前，先問每格究竟是什麼單位。

短程式沿用本課工具與原計算語義。安裝、長訓練與重做操作見[訓練配方](https://birdhackor.github.io/tiny-perceptron-vlm/training.html)，不需要先完成長配方才能閱讀這個例子。

</details>
